# Nooble — MiniMax H3 Google Colab — AI Video Generation & LoRA

Generate video with sound using native ComfyUI nodes. Choose a preset, optionally load a compatible model-only LoRA, then run Steps 1–5 in order. **When finished, enable STOP_BACKEND in Step 6, run it, then disconnect and delete the runtime.**

Use an **A100 GPU with high RAM** and roughly **50 GB of free storage** for one model set and headroom. GPU allocation and Colab compute are not guaranteed. The original FL2VA 8-step notebook generated a clip; these refactored presets and custom-LoRA paths still need GPU validation.

[Repository and help](https://github.com/narcisoJavier/Nooble) · [MiniMax H3 guide](https://docs.comfy.org/tutorials/video/minimax/minimax-h3)


In [ ]:
# @title Step 1 — Choose model, cache, and optional LoRA
# @markdown Choose before setup. FL2VA supports text/first-frame generation; Ref2VA uses a reference image. Changing these fields requires rerunning Steps 1-4.
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
PRESET = "fl2va-turbo-8"  # @param ["fl2va-turbo-8", "fl2va-base", "fl2va-turbo-4", "ref2va-base", "ref2va-turbo-4"]
# @markdown Optional compatible H3 model-only LoRA: full Hugging Face file URL or absolute Drive `.safetensors` path. Leave blank for none. Use the matching FL2VA/Ref2VA family. This loads adapters; it does not train them.
CUSTOM_LORA = ""  # @param {type:"string"}

import platform
import subprocess
from pathlib import Path

if platform.system() != 'Linux' or not Path('/content').is_dir():
    raise RuntimeError('Run this notebook in a Google Colab GPU runtime.')
try:
    gpu = subprocess.check_output([
        'nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'
    ], text=True).strip().splitlines()[0]
except (FileNotFoundError, subprocess.CalledProcessError, IndexError) as exc:
    raise RuntimeError('Select Runtime > Change runtime type > A100 GPU.') from exc
gpu_name, gpu_mib = gpu.rsplit(',', 1)
ram_kib = int(next(line.split()[1] for line in Path('/proc/meminfo').read_text().splitlines()
                   if line.startswith('MemTotal:')))
if 'A100' not in gpu_name or int(gpu_mib.strip()) < 38000:
    raise RuntimeError(f'This notebook targets an A100 with at least 38,000 MiB VRAM; found {gpu}.')
if ram_kib / 1024**2 < 48:
    raise RuntimeError('Select a high-RAM runtime with at least 48 GiB system RAM.')
print(f'GPU: {gpu_name.strip()}, {int(gpu_mib.strip()) / 1024:.1f} GiB VRAM')
print(f'System RAM: {ram_kib / 1024**2:.1f} GiB')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI_Models')
    OUTPUT_DIR = Path('/content/drive/MyDrive/ComfyUI_Outputs')
else:
    MODELS_DIR = Path('/content/H3_models')
    OUTPUT_DIR = Path('/content/H3_outputs')
    print('Runtime-only storage selected: download outputs before deleting the runtime.')
for directory in (MODELS_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
COMFY_DIR = Path('/content/ComfyUI_H3')
prepared = None
backend_prepared = None
print('Model cache:', MODELS_DIR)
print('Videos:', OUTPUT_DIR)


In [ ]:
# @title Step 2 — Install the pinned notebook runner and ComfyUI
import sys

RELEASE = 'v0.2.0'
REPO_DIR = Path('/content/minimax-h3-colab')
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', RELEASE,
        'https://github.com/narcisoJavier/Nooble.git', str(REPO_DIR)
    ], check=True)
head = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
release_head = subprocess.check_output([
    'git', '-C', str(REPO_DIR), 'rev-parse', f'refs/tags/{RELEASE}^{{commit}}'
], text=True).strip()
if head != release_head:
    raise RuntimeError('Existing runner checkout is a different version. Use a fresh runtime.')
if subprocess.run([
    'git', '-C', str(REPO_DIR), 'diff', '--quiet', 'HEAD', '--', 'runner.py', 'presets.json'
]).returncode:
    raise RuntimeError('Existing runner files were modified. Use a fresh runtime.')
sys.path.insert(0, str(REPO_DIR))
import runner
if Path(runner.__file__).resolve() != REPO_DIR / 'runner.py':
    raise RuntimeError('Another runner module is loaded. Restart the runtime and run in order.')
runner.install_comfy(COMFY_DIR)
print('Runner release:', RELEASE)


In [ ]:
# @title Step 3 — Download or reuse the selected model files
# @markdown Public model downloads need no token. For a gated/private custom LoRA, save `HF_TOKEN` in Colab Secrets (key icon) and grant this notebook access. Never paste a token into a URL or form.
token = None
if CUSTOM_LORA.strip().startswith('https://huggingface.co/'):
    from google.colab import userdata
    try:
        token = userdata.get('HF_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print('No accessible HF_TOKEN; trying the custom LoRA as a public download.')
try:
    custom_lora_name = runner.prepare_models(
        PRESET, MODELS_DIR, custom_lora=CUSTOM_LORA, token=token
    )
finally:
    token = None
prepared = (PRESET, CUSTOM_LORA)
preset_info = runner.CATALOG['presets'][PRESET]
print('Preset:', preset_info['label'])
print('Steps:', preset_info['steps'], '| Sampler:', preset_info['sampler'])
print('Sigma shifts — video:', preset_info['shift_video'], 'audio:', preset_info['shift_audio'])
if PRESET == 'ref2va-turbo-4':
    print('Experimental Ref2VA Turbo 4: trained at 544p. Try WIDTH=960, HEIGHT=544.')


In [ ]:
# @title Step 4 — Start ComfyUI and wait for readiness
if globals().get('prepared') != (PRESET, CUSTOM_LORA):
    raise RuntimeError('Run Step 3 for the selected preset and LoRA first.')
comfy_proc = runner.start_backend(
    COMFY_DIR, MODELS_DIR, OUTPUT_DIR, previous=globals().get('comfy_proc')
)

backend_prepared = prepared


In [ ]:
# @title Step 5 — Prompt and generate video
# @markdown Default: 1344×768, approximately 5.17 seconds. Ref2VA requires `INPUT_IMAGE` and a prompt referring to `<Picture 1>`. FL2VA can optionally use the image as its first frame. Experimental Ref2VA Turbo 4 was trained at 544p; try 960×544.
PROMPT = "Cinematic live-action wildlife shot, one continuous take. A small red panda sits on a moss-covered branch in a misty bamboo forest at sunrise. It gently turns toward the camera, blinks, and tilts its head while a light breeze moves its fur and the bamboo leaves. Camera: a slow, steady push-in from a medium shot to a close-up, shallow depth of field, warm golden rim light, realistic fur and natural movement. Audio: soft wind rustling bamboo, distant birds, and quiet forest ambience; no dialogue or music. No cuts, captions, text, or logos."  # @param {type:"string"}
WIDTH = 1344  # @param {type:"integer"}
HEIGHT = 768  # @param {type:"integer"}
SECONDS = 5  # @param {type:"number"}
SEED = 12345  # @param {type:"integer"}
# @markdown Local PNG/JPEG/WebP path, for example `/content/drive/MyDrive/reference.png`. Leave blank for text-only FL2VA.
INPUT_IMAGE = ""  # @param {type:"string"}
CUSTOM_LORA_STRENGTH = 1.0  # @param {type:"number"}
TIMEOUT_MINUTES = 60  # @param {type:"integer"}

from IPython.display import Video, display

if (globals().get('prepared') != (PRESET, CUSTOM_LORA)
        or globals().get('backend_prepared') != (PRESET, CUSTOM_LORA)):
    raise RuntimeError('Run Steps 3 and 4 for the selected preset and LoRA before generating.')
last_video = runner.generate(
    PRESET, PROMPT, globals().get('comfy_proc'), COMFY_DIR, OUTPUT_DIR,
    image_path=INPUT_IMAGE, timeout_minutes=TIMEOUT_MINUTES,
    width=WIDTH, height=HEIGHT, seconds=SECONDS, seed=SEED,
    lora_name=custom_lora_name, lora_strength=CUSTOM_LORA_STRENGTH
)
display(Video(str(last_video), embed=True))

# Optional download to your computer:
# from google.colab import files
# files.download(str(last_video))


In [ ]:
# @title Step 6 — Stop when finished generating
# @markdown Enable this only when finished. Then use Runtime > Disconnect and delete runtime to release the GPU.
STOP_BACKEND = False  # @param {type:"boolean"}
if STOP_BACKEND:
    runner.stop_backend(globals().get('comfy_proc'))
    print('ComfyUI stopped. Saved videos:', OUTPUT_DIR)
    if not USE_GOOGLE_DRIVE:
        print('Download videos before deleting the runtime; /content storage is temporary.')
    print('Release the GPU: Runtime > Disconnect and delete runtime.')
else:
    print('Backend left running. Enable STOP_BACKEND when finished, then delete the runtime.')
